<h2> Query and Performance Analysis</h2>

<h5> Setup

In [1]:
import pandas as pd
import pymysql
import getpass # May need to import as `from getpass import getpass`
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
import sqlite3

In [2]:
conn = sqlite3.connect("flights.db")

In [3]:
airlines = pd.read_csv("airlines.csv")
airports = pd.read_csv("airports.csv")
flights = pd.read_csv("flights.csv", low_memory=False)

In [4]:
airlines.to_sql("airlines", conn, if_exists="replace", index=False)
airports.to_sql("airports", conn, if_exists="replace", index=False)
flights.to_sql("flights", conn, if_exists="replace", index=False) #big dataset, takes a while to load

5819079

Query 1: Departure Avg Aggregation

Case 1: No Indexes

In [5]:
conn.execute("DROP INDEX IF EXISTS idx_airline;")
pd.set_option('display.max_colwidth', None)


In [6]:
%%time
pd.read_sql_query('''
            EXPLAIN QUERY PLAN
            SELECT a.AIRLINE, AVG(f.DEPARTURE_DELAY) AS Average_Delay
            FROM flights f
            JOIN airlines a ON a.IATA_CODE = f.AIRLINE 
            WHERE DEPARTURE_DELAY > 0 AND DEPARTURE_DELAY IS NOT NULL 
                AND DEPARTURE_TIME IS NOT NULL 
                AND ARRIVAL_TIME IS NOT NULL
            GROUP BY a.AIRLINE
            ORDER BY AVG(f.DEPARTURE_DELAY) ASC
            ;''', conn)

CPU times: user 939 μs, sys: 2.18 ms, total: 3.12 ms
Wall time: 2.65 ms


,id,parent,notused,detail
0,8,0,216,SCAN f
1,31,0,53,SEARCH a USING AUTOMATIC COVERING INDEX (IATA_CODE=?)
2,36,0,0,USE TEMP B-TREE FOR GROUP BY
3,76,0,0,USE TEMP B-TREE FOR ORDER BY


Case 2: Index on flights(AIRLINES)

In [7]:
conn.execute("DROP INDEX IF EXISTS idx_airline;")
conn.execute("""CREATE INDEX idx_airline
ON flights(AIRLINE);
""")


In [8]:
%%time
pd.read_sql_query('''
            EXPLAIN QUERY PLAN
            SELECT a.AIRLINE, AVG(f.DEPARTURE_DELAY) AS Average_Delay
            FROM flights f
            JOIN airlines a ON a.IATA_CODE = f.AIRLINE 
            WHERE DEPARTURE_DELAY > 0 AND DEPARTURE_DELAY IS NOT NULL 
                AND DEPARTURE_TIME IS NOT NULL 
                AND ARRIVAL_TIME IS NOT NULL
            GROUP BY a.AIRLINE
            ORDER BY AVG(f.DEPARTURE_DELAY) ASC
            ;''', conn)

CPU times: user 738 μs, sys: 166 μs, total: 904 μs
Wall time: 871 μs


,id,parent,notused,detail
0,9,0,216,SCAN a
1,11,0,61,SEARCH f USING INDEX idx_airline (AIRLINE=?)
2,29,0,0,USE TEMP B-TREE FOR GROUP BY
3,69,0,0,USE TEMP B-TREE FOR ORDER BY


Query 2: Delay Analysis Query 

Case 1: No Index

In [9]:
conn.execute("DROP INDEX IF EXISTS idx_airline;")

In [10]:
%%time
pd.read_sql_query('''
        EXPLAIN QUERY PLAN
            select
    airline,
    avg(departure_delay) as avg_departure_delay
from flights
where cancelled = 0
group by airline
order by avg_departure_delay desc
            ;''', conn)

CPU times: user 1.32 ms, sys: 738 μs, total: 2.06 ms
Wall time: 2.23 ms


,id,parent,notused,detail
0,7,0,216,SCAN flights
1,11,0,0,USE TEMP B-TREE FOR GROUP BY
2,50,0,0,USE TEMP B-TREE FOR ORDER BY


Case 2: Index on flight(AIRLINES)

In [11]:
conn.execute("DROP INDEX IF EXISTS idx_airline;")
conn.execute("""CREATE INDEX idx_airline
ON flights(AIRLINE);
""")

In [12]:
%%time
pd.read_sql_query('''
        EXPLAIN QUERY PLAN
            select
    airline,
    avg(departure_delay) as avg_departure_delay
from flights
where cancelled = 0
group by airline
order by avg_departure_delay desc
            ;''', conn)

CPU times: user 952 μs, sys: 641 μs, total: 1.59 ms
Wall time: 1.14 ms


,id,parent,notused,detail
0,8,0,223,SCAN flights USING INDEX idx_airline
1,42,0,0,USE TEMP B-TREE FOR ORDER BY


Case 3: Adding another Index on flights(CANCELLED)

In [13]:
conn.execute("DROP INDEX IF EXISTS idx_airline;")
conn.execute("DROP INDEX IF EXISTS idx_cancelled;")
conn.execute("""CREATE INDEX idx_airline
ON flights(AIRLINE);
""")
conn.execute("""CREATE INDEX idx_cancelled
ON flights(CANCELLED);
""")

In [ ]:
%%time
pd.read_sql_query('''
        EXPLAIN QUERY PLAN
            select
    airline,
    avg(departure_delay) as avg_departure_delay
from flights
where cancelled = 0
group by airline
order by avg_departure_delay desc
            ;''', conn)

#indexing on a low cardinality column like cancelled is worse for performance. no index is better here

CPU times: user 1.11 ms, sys: 1.31 ms, total: 2.42 ms
Wall time: 1.86 ms


,id,parent,notused,detail
0,8,0,60,SEARCH flights USING INDEX idx_cancelled (CANCELLED=?)
1,13,0,0,USE TEMP B-TREE FOR GROUP BY
2,52,0,0,USE TEMP B-TREE FOR ORDER BY


Query 3: MongoDB query on Avg Arrivals Delay

In [15]:
import pymongo
from pymongo import MongoClient
import pandas as pd
client = MongoClient('localhost', 27017)
db = client.example

In [16]:
flights_airlines = pd.merge(airlines, flights, left_on='IATA_CODE', right_on = 'AIRLINE', how = 'inner' )

In [23]:
sample = flights_airlines.sample(10000, random_state=1)
sample_dict = sample.to_dict('records')
flights_airlines_collection = db.flights_airlines
result0 = flights_airlines_collection.insert_many(sample_dict)

In [25]:
pipeline = [
    {'$match': {'ARRIVAL_DELAY': {'$gt': 0, '$ne': None},
            'ARRIVAL_TIME': {'$ne': None},
            'DIVERTED': 0,
            'CANCELLED': 0}},
    {'$group': {'_id': '$AIRLINE_x', 
                'Average_Arrival_Delay': {'$avg': '$ARRIVAL_DELAY'}}},
    {'$sort': {'Average_Arrival_Delay': 1}}
]
result = flights_airlines_collection.aggregate(pipeline)

print(db.command('aggregate', 'flights_airlines', pipeline=pipeline, explain=True))


{'explainVersion': '2', 'stages': [{'$cursor': {'queryPlanner': {'namespace': 'example.flights_airlines', 'parsedQuery': {'$and': [{'CANCELLED': {'$eq': 0}}, {'DIVERTED': {'$eq': 0}}, {'ARRIVAL_DELAY': {'$gt': 0}}, {'ARRIVAL_DELAY': {'$not': {'$eq': None}}}, {'ARRIVAL_TIME': {'$not': {'$eq': None}}}]}, 'indexFilterSet': False, 'queryHash': 'C7B67C55', 'planCacheShapeHash': 'C7B67C55', 'planCacheKey': 'FA6D260E', 'optimizationTimeMillis': 0, 'cursorType': 'regular', 'maxIndexedOrSolutionsReached': False, 'maxIndexedAndSolutionsReached': False, 'maxScansToExplodeReached': False, 'prunedSimilarIndexes': False, 'winningPlan': {'isCached': False, 'queryPlan': {'stage': 'GROUP', 'planNodeId': 3, 'inputStage': {'stage': 'COLLSCAN', 'planNodeId': 1, 'filter': {'$and': [{'CANCELLED': {'$eq': 0}}, {'DIVERTED': {'$eq': 0}}, {'ARRIVAL_DELAY': {'$gt': 0}}, {'ARRIVAL_DELAY': {'$not': {'$eq': None}}}, {'ARRIVAL_TIME': {'$not': {'$eq': None}}}]}, 'nss': 'example.flights_airlines', 'direction': 'forwar

'winningPlan': {
    'queryPlan': {
        'stage': 'GROUP',
        'inputStage': {
            'stage': 'COLLSCAN',

Analyze this later